In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# 古典フィードフォワードと制御フロー（動的回路）

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



# 古典フィードフォワードと制御フロー


<details>
<summary><b>パッケージバージョン</b></summary>

このページのコードは以下の要件を使用して開発されました。
これらのバージョン以降を使用することを推奨します。

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
> **Note:** 動的回路の新バージョンが、すべてのユーザーのすべてのバックエンドで利用可能になりました。ユーティリティスケールで動的回路を実行できるようになりました。詳細は[お知らせ](/announcements/product-updates/2025-09-25-new-dynamic-circuits)をご覧ください。

動的回路は強力なツールであり、量子回路の実行中にQubitを測定し、その中回路測定の結果に基づいて回路内で古典的な論理演算を実行することができます。このプロセスは_古典フィードフォワード_とも呼ばれます。動的回路を最大限に活用する方法についてはまだ初期段階ですが、量子研究コミュニティはすでにいくつかのユースケースを特定しています。例えば以下のものがあります：

* [GHZ状態、](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339) [W状態、](https://arxiv.org/abs/2403.07604)（W状態の詳細については、["フィードフォワードを使用した浅い回路による状態準備"](https://arxiv.org/abs/2307.14840)も参照してください）、および幅広い[行列積状態](https://arxiv.org/abs/2404.16083)などの効率的な量子状態準備
* 浅い回路を使用した同じチップ上のQubit間の[効率的な長距離エンタングルメント](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339)
* [IQP型回路の効率的なサンプリング](https://arxiv.org/pdf/2505.04705)

しかし、動的回路がもたらすこれらの改善にはトレードオフが伴います。中回路測定と古典演算は、通常、2-Qubitゲートよりも実行時間が長く、この時間の増加が回路深さの削減による利点を相殺する可能性があります。そのため、IBM Quantum&reg;が動的回路の[新バージョン](/announcements/product-updates/2025-03-03-new-version-dynamic-circuits)をリリースする中、中回路測定の時間短縮は改善の重点分野となっています。

[OpenQASM 3仕様](https://openqasm.com/language/classical.html#looping-and-branching)では多くの制御フロー構造が定義されていますが、Qiskit Runtimeは現在、条件付き`if`文のみをサポートしています。Qiskit SDKでは、これは[QuantumCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit)の[if_test](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#if_test)メソッドに対応します。このメソッドは[コンテキストマネージャー](https://docs.python.org/3/reference/datamodel.html#with-statement-context-managers)を返し、通常は`with`文で使用されます。このガイドでは、この条件文の使用方法について説明します。

> **Note:** このガイドのコード例では、中回路測定に標準のmeasure命令を使用しています。ただし、バックエンドがサポートしている場合は、代わりに[`MidCircuitMeasure`](/guides/measure-qubits#midcircuit)命令を使用することを推奨します。詳細は[中回路測定のドキュメント](/guides/measure-qubits#mid-circuit-measurements)をご覧ください。
## `if`文
`if`文は、古典ビットまたはレジスタの値に基づいて条件付きで演算を実行するために使用します。

以下の例では、QubitにHadamardゲートを適用して測定します。結果が1の場合、QubitにXゲートを適用します。これにより、Qubitを0の状態に戻します。次に、Qubitを再度測定します。結果として得られる測定結果は、100%の確率で0になるはずです。

In [1]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
circuit = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

circuit.h(q0)
# Use MidCircuitMeasure() if it's supported by the backend.
# circuit.append(MidCircuitMeasure(), [q0], [c0])
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)):
    circuit.x(q0)
circuit.measure(q0, c0)
circuit.draw("mpl")

# example output counts: {'0': 1024}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/60924bfa-50ed-4d9d-a17b-9d64f2cc053f-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/60924bfa-50ed-4d9d-a17b-9d64f2cc053f-0.svg)

`with`文には代入ターゲットを指定することができます。これ自体もコンテキストマネージャーであり、格納してその後elseブロックの作成に使用できます。elseブロックは、`if`ブロックの内容が*実行されない*場合に常に実行されます。

以下の例では、2つのQubitと2つの古典ビットのレジスタを初期化します。最初のQubitにHadamardゲートを適用して測定します。結果が1の場合、2番目のQubitにHadamardゲートを適用します。それ以外の場合は、2番目のQubitにXゲートを適用します。最後に、2番目のQubitも測定します。

In [2]:
qubits = QuantumRegister(2)
clbits = ClassicalRegister(2)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1) = qubits
(c0, c1) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)) as else_:
    circuit.h(q1)
with else_:
    circuit.x(q1)
circuit.measure(q1, c1)

circuit.draw("mpl")

# example output counts: {'01': 260, '11': 272, '10': 492}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/20f0640a-a3f7-41b3-aada-b66bc89b0555-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/20f0640a-a3f7-41b3-aada-b66bc89b0555-0.svg)

単一の古典ビットを条件とするだけでなく、複数のビットで構成される古典レジスタの値を条件とすることも可能です。

以下の例では、2つのQubitにHadamardゲートを適用して測定します。結果が`01`（つまり、最初のQubitが1で2番目のQubitが0）の場合、3番目のQubitにXゲートを適用します。最後に、3番目のQubitを測定します。なお、わかりやすくするために、`if`条件で3番目の古典ビットの状態（0）を明示的に指定しています。回路図では、条件は条件付けされている古典ビット上の丸で示されます。黒い丸は1への条件付けを示し、白い丸は0への条件付けを示します。

In [3]:
qubits = QuantumRegister(3)
clbits = ClassicalRegister(3)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1, q2) = qubits
(c0, c1, c2) = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.if_test((clbits, 0b001)):
    circuit.x(q2)
circuit.measure(q2, c2)

circuit.draw("mpl")

# example output counts: {'101': 269, '011': 260, '000': 252, '010': 243}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/98e8f552-4169-42a3-8182-e14e9ffb59e2-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/98e8f552-4169-42a3-8182-e14e9ffb59e2-0.svg)

## 古典式
Qiskitの古典式モジュール[`qiskit.circuit.classical`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical)には、回路実行中に古典値に対するランタイム演算の試験的な表現が含まれています。ハードウェアの制限により、現在は`QuantumCircuit.if_test()`の条件のみがサポートされています。

以下の例では、パリティの計算を使用して動的回路でn-QubitのGHZ状態を作成する方法を示します。まず、隣接するQubitに$n/2$個のBellペアを生成します。次に、ペア間のCNOTゲート層を使用してこれらのペアを結合します。その後、以前のすべてのCNOTゲートのターゲットQubitを測定し、測定された各Qubitを状態$\vert 0 \rangle$にリセットします。先行するすべてのビットのパリティが奇数である測定されていないすべてのサイトに$X$を適用します。最後に、測定されたQubitにCNOTゲートを適用して、測定で失われたエンタングルメントを再確立します。

パリティ計算では、構築された式の最初の要素はPythonオブジェクト`mr[0]`を[`Value`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical#value)ノードに持ち上げることを含みます（`lift`は任意のオブジェクトを古典式に変換するために使用されます）。`mr[1]`と後続の可能な古典レジスタについては、これは必要ありません。これらは`expr.bit_xor`への入力であり、必要な持ち上げはこれらの場合に自動的に行われるためです。このような式はループやその他の構造で構築できます。

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.classical import expr

num_qubits = 8
if num_qubits % 2 or num_qubits < 4:
    raise ValueError("num_qubits must be an even integer ≥ 4")
meas_qubits = list(range(2, num_qubits, 2))  # qubits to measure and reset

qr = QuantumRegister(num_qubits, "qr")
mr = ClassicalRegister(len(meas_qubits), "m")
qc = QuantumCircuit(qr, mr)

# Create local Bell pairs
qc.reset(qr)
qc.h(qr[::2])
for ctrl in range(0, num_qubits, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Glue neighboring pairs
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Measure boundary qubits between pairs,reset to 0
for k, q in enumerate(meas_qubits):
    qc.measure(qr[q], mr[k])
    qc.reset(qr[q])

# Parity-conditioned X corrections
# Each non-measured qubit gets flipped iff the parity (XOR) of all
# preceding measurement bits is 1
for tgt in range(num_qubits):
    if tgt in meas_qubits:  # skip measured qubits
        continue
    # all measurement registers whose physical qubit index < tgt
    left_bits = [k for k, q in enumerate(meas_qubits) if q < tgt]
    if not left_bits:  # skip if list empty
        continue

    # build XOR-parity expression
    parity = expr.lift(
        mr[left_bits[0]]
    )  # lift the first bit to Value so it will be treated like a boolean.
    for k in left_bits[1:]:
        parity = expr.bit_xor(
            mr[k], parity
        )  # calculate parity with all other bits
    with qc.if_test(parity):  # Add X if parity is 1
        qc.x(qr[tgt])

# Re-entangle measured qubits
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

In [5]:
qc.draw(output="mpl", style="iqp", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d0f0abdb-50d5-408d-a704-a1a555acdd85-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d0f0abdb-50d5-408d-a704-a1a555acdd85-0.svg)

## 動的回路をサポートするバックエンドを見つける
アカウントがアクセスでき、動的回路をサポートするすべてのバックエンドを見つけるには、以下のようなコードを実行します。この例では、[ログイン資格情報を保存済み](/guides/save-credentials)であることを前提としています。Qiskit Runtimeサービスアカウントを初期化する際に[明示的に資格情報を指定する](/guides/initialize-account#explicit)こともできます。これにより、特定のインスタンスやプランタイプで利用可能なバックエンドを表示できます。

> **Note:** - アカウントで利用可能なバックエンドは、資格情報で指定されたインスタンスによって異なります。
> - 動的回路の新バージョンが、すべてのユーザーのすべてのバックエンドで利用可能になりました。詳細は[お知らせ](/announcements/product-updates/2025-09-25-new-dynamic-circuits)をご覧ください。

In [6]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
dc_backends = service.backends(dynamic_circuits=True)
print(dc_backends)

[<IBMBackend('ibm_pittsburgh')>, <IBMBackend('ibm_boston')>, <IBMBackend('ibm_fez')>, <IBMBackend('ibm_miami')>, <IBMBackend('ibm_marrakesh')>, <IBMBackend('ibm_torino')>, <IBMBackend('ibm_kingston')>]


## Qiskit Runtime limitations

Be aware of the following constraints when running dynamic circuits in Qiskit Runtime.

- Due to the limited physical memory on control electronics, there is also a limit on the number of `if` statements and size of their operands. This limit is a function of the number of broadcasts and the number of broadcasted bits in a job (not a circuit).

   When processing an `if` condition, measurement data needs to be transferred to the control logic to make that evaluation. A broadcast is a transfer of unique classical data, and broadcasted bits is the number of classical bits being transferred. Consider the following:

   ```python
   c0 = ClassicalRegister(3)
   c1 = ClassicalRegister(5)
   ...
   with circuit.if_test((c0, 1)) ...
   with circuit.if_test((c0, 3)) ...
   with circuit.if_test((c1[2], 1)) ...
   ```
   In the previous code example, the first two `if_test` objects on `c0` are considered one broadcast because the content of `c0` did not change, and thus does not need to be re-broadcasted. The `if_test` on `c1` is a second broadcast. The first one broadcasts all three bits in `c0` and the second broadcasts just one bit, making a total of four broadcasted bits.

   Currently, if you broadcast 60 bits each time, then the job can have approximately 300 broadcasts. If you broadcast just one bit each time, however, then the job can have 2400 broadcasts.

- The operand used in an `if_test` statement must be 32 or fewer bits. Thus, if you are comparing an entire `ClassicalRegister`, the size of that `ClassicalRegister` must be 32 or fewer bits. If you are comparing just a single bit from a `ClassicalRegister`, however, that `ClassicalRegister` can be of any size (since the operand is only one bit).

   For example, the "Not valid" code block does not work because `cr` is more than 32 bits.  You can, however, use a classical register wider than 32 bits if you are testing only one bit, as shown in the "Valid" code block.

   <Tabs>
   <TabItem value="Not valid" label="Not valid">
      ```python
         cr = ClassicalRegister(50)
         qr = QuantumRegister(50)
         circuit = QuantumCircuit(qr, cr)
         ...
         circ.measure(qr, cr)
         with circ.if_test((cr, 15)):
            ...
      ```
   </TabItem>
   <TabItem value="Valid" label="Valid">
      ```python
         cr = ClassicalRegister(50)
         qr = QuantumRegister(50)
         circuit = QuantumCircuit(qr, cr)
         ...
         circ.measure(qr, cr)
         with circ.if_test((cr[5], 1)):
            ...
      ```
   </TabItem>
   </Tabs>

- Nested conditionals are not allowed. For example, the following code block will not work because it has an `if_test` inside another `if_test`:
   <Tabs>
    <TabItem value="Not valid" label="Not valid">
        ```python
           c1 = ClassicalRegister(1, "c1")
           c2 = ClassicalRegister(2, "c2")
           ...
           with circ.if_test((c1, 1)):
            with circ.if_test(c2, 1)):
             ...
        ```
     </TabItem>
     <TabItem value="Valid" label="Valid">
        ```python
        cr = ClassicalRegister(2)
        ...
        with circuit.if_test((cr, 0b11)):
          ...
        ```
     </TabItem>
    </Tabs>

- Having `reset` or measurements inside conditionals is not supported.
- Arithmetic operations are not supported.
- See the [OpenQASM 3 feature table](/docs/guides/qasm-feature-table) to determine which OpenQASM 3 features are supported on Qiskit and Qiskit Runtime.
- When OpenQASM 3 (instead of `QuantumCircuit`) is used as the input format to pass circuits to Qiskit Runtime primitives, only instructions that can be loaded into Qiskit are supported. Classical operations, for example, are not supported because they cannot be loaded into Qiskit. See [Import an OpenQASM 3 program into Qiskit](/docs/guides/interoperate-qiskit-qasm3#import-an-openqasm-3-program-into-qiskit) for more information.
- The `for`, `while`, and `switch` instructions are not supported.

## Use dynamic circuits with Estimator

Since Estimator does not support dynamic circuits, you can use Sampler and build your own measurement circuits instead. Alternatively, you can use the [Executor primitive,](/docs/guides/directed-execution-model#executor-primitive) which supports dynamic circuits.

To replicate Estimator's behavior, follow this process:

1. Group the terms of all observables into a partition.  This can be done by using the [`PauliList` API,](/docs/api/qiskit/qiskit.quantum_info.PauliList#group_qubit_wise_commuting) for example.
     <Admonition type="note">
      You can use the [`BitArray`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.primitives.BitArray#expectation_values) primitive attribute to compute the expectation values of the provided observables.
     </Admonition>
2. Execute one basis change circuit per partition (whichever basis change needs to be done for each partition). See the Measurement bases addon utility  [`measurement_bases` module](https://github.com/Qiskit/qiskit-addon-utils/blob/38ea05431f2e9830adf4ec9265f6d19758a32096/qiskit_addon_utils/exp_vals/measurement_bases.py) for more information. [Get started with utilities.](/docs/guides/qiskit-addons-utils#get-started-with-utilities)
3. Add back together the results for each partition.

## Next steps

<Admonition type="tip" title="Recommendations">
- Learn how to implement accurate dynamic decoupling by using [stretch.](/docs/guides/stretch)
- Learn about the shorter [mid-circuit measurements](/docs/guides/measure-qubits#mid-circuit-measurements) that reduce the circuit time.
- Use [circuit schedule visualization](/docs/guides/visualize-circuit-timing#qiskit-runtime-support) to debug and optimize your dynamic circuits.
</Admonition>